# Part 2 — Data Cleaning

**Week 8: E-Commerce Order Analytics System**

data-cleaning tasks

- Clean orders
- Fix date formats
- Handle missing customer IDs
- Normalize product names
- Validate customer emails
- Check referential integrity
- Detect negative and zero quantities
- Generate a data-quality report
- Save cleaned CSV files

In [1]:
import pandas as pd
import os
import re

In [2]:
# ==========================================
# File Paths
# ==========================================

ORDERS_FILE = "data/orders.csv"
ORDER_ITEMS_FILE = "data/order_items.csv"
PRODUCTS_FILE = "data/products.csv"
CUSTOMERS_FILE = "data/customers.csv"

CLEANED_ORDERS_FILE = "data/cleaned_orders.csv"
CLEANED_ORDER_ITEMS_FILE = "data/cleaned_order_items.csv"
CLEANED_PRODUCTS_FILE = "data/cleaned_products.csv"
CLEANED_CUSTOMERS_FILE = "data/cleaned_customers.csv"

REPORT_FILE = "reports/data_quality_report.txt"

# Store all issues
issues = []

## 1. Clean Orders

In [3]:
def clean_orders():

    print("\nCleaning orders...")

    df = pd.read_csv(ORDERS_FILE)
    original_count = len(df)

    # Handle NULL customer IDs
    null_customer_count = df["customer_id"].isna().sum()

    if null_customer_count > 0:
        issues.append(
            f"Orders: {null_customer_count} rows have missing customer_id."
        )
        df["customer_id"] = df["customer_id"].fillna("UNKNOWN")

    # Fix order dates
    invalid_date_count = 0

    def fix_date(date_value):
        nonlocal invalid_date_count

        if pd.isna(date_value):
            return None

        date_value = str(date_value).strip()

        # Standard YYYY-MM-DD format
        try:
            return pd.to_datetime(
                date_value,
                format="%Y-%m-%d %H:%M:%S"
            )
        except ValueError:
            pass

        # DD-MM-YYYY format
        try:
            invalid_date_count += 1
            return pd.to_datetime(
                date_value,
                format="%d-%m-%Y %H:%M:%S"
            )
        except ValueError:
            issues.append(
                f"Orders: Invalid date found: {date_value}"
            )
            return pd.NaT

    df["order_date"] = df["order_date"].apply(fix_date)

    if invalid_date_count > 0:
        issues.append(
            f"Orders: {invalid_date_count} dates were converted "
            f"from DD-MM-YYYY format."
        )

    df["order_date"] = df["order_date"].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    # Check duplicate order IDs
    duplicate_orders = df["order_id"].duplicated().sum()

    if duplicate_orders > 0:
        issues.append(
            f"Orders: {duplicate_orders} duplicate order_id values found."
        )
        df = df.drop_duplicates(subset=["order_id"])

    # Save cleaned orders
    df.to_csv(CLEANED_ORDERS_FILE, index=False)

    print(f"Original orders : {original_count}")
    print(f"Cleaned orders  : {len(df)}")

    return df

## 2. Clean Products

In [5]:
def clean_products():

    print("\nCleaning products...")

    df = pd.read_csv(PRODUCTS_FILE)

    # Check missing product names
    missing_names = df["product_name"].isna().sum()

    if missing_names > 0:
        issues.append(
            f"Products: {missing_names} missing product names found."
        )
        df["product_name"] = df["product_name"].fillna(
            "Unknown Product"
        )

    # Normalize product names
    df["product_name"] = (
        df["product_name"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    issues.append(
        "Products: Product names were trimmed and converted to title case."
    )

    # Check duplicate product IDs
    duplicate_products = df["product_id"].duplicated().sum()

    if duplicate_products > 0:
        issues.append(
            f"Products: {duplicate_products} duplicate "
            f"product_id values found."
        )
        df = df.drop_duplicates(subset=["product_id"])

    # Save cleaned products
    df.to_csv(CLEANED_PRODUCTS_FILE, index=False)

    print(f"Products cleaned: {len(df)}")

    return df

## 3. Validate Emails

In [6]:
def validate_emails():

    print("\nValidating emails...")

    df = pd.read_csv(CUSTOMERS_FILE)

    invalid_customer_ids = []

    email_pattern = (
        r"^[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    )

    for _, row in df.iterrows():

        customer_id = row["customer_id"]
        email = row["email"]

        if pd.isna(email):
            invalid_customer_ids.append(customer_id)

        elif not re.match(
            email_pattern,
            str(email).strip()
        ):
            invalid_customer_ids.append(customer_id)

    if invalid_customer_ids:
        issues.append(
            f"Customers: {len(invalid_customer_ids)} "
            f"invalid email addresses found."
        )

    print(f"Invalid emails: {len(invalid_customer_ids)}")

    # Save customers
    df.to_csv(CLEANED_CUSTOMERS_FILE, index=False)

    return invalid_customer_ids

## 4. Check Referential Integrity

In [7]:
def check_referential_integrity():

    print("\nChecking referential integrity...")

    orders = pd.read_csv(CLEANED_ORDERS_FILE)
    order_items = pd.read_csv(ORDER_ITEMS_FILE)

    valid_order_ids = set(orders["order_id"])

    invalid_items = order_items[
        ~order_items["order_id"].isin(valid_order_ids)
    ]

    invalid_count = len(invalid_items)

    if invalid_count > 0:
        issues.append(
            f"Order Items: {invalid_count} items reference "
            f"non-existent orders."
        )
        print(f"Invalid order references: {invalid_count}")
    else:
        print("All order_items reference valid orders.")

    # Negative quantities represent returns
    negative_quantity_count = (
        order_items["quantity"] < 0
    ).sum()

    if negative_quantity_count > 0:
        issues.append(
            f"Order Items: {negative_quantity_count} negative "
            f"quantities found (treated as returns)."
        )

    # Zero quantities
    zero_quantity_count = (
        order_items["quantity"] == 0
    ).sum()

    if zero_quantity_count > 0:
        issues.append(
            f"Order Items: {zero_quantity_count} rows "
            f"have quantity = 0."
        )

    # Save cleaned order items
    order_items.to_csv(
        CLEANED_ORDER_ITEMS_FILE,
        index=False
    )

    return invalid_items

## 5. Generate Data Quality Report

In [8]:
def generate_report():

    os.makedirs("reports", exist_ok=True)

    with open(
        REPORT_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        file.write("E-COMMERCE ORDER ANALYTICS\n")
        file.write("DATA QUALITY REPORT\n")
        file.write("=" * 50 + "\n\n")

        if issues:
            for i, issue in enumerate(issues, start=1):
                file.write(f"{i}. {issue}\n")
        else:
            file.write("No data quality issues found.\n")

    print(f"\nReport created: {REPORT_FILE}")

## 6. Run the Complete Data Cleaning Pipeline

In [9]:
print("=" * 60)
print("E-COMMERCE DATA CLEANING")
print("=" * 60)

# Clean orders
orders_cleaned = clean_orders()

# Clean products
products_cleaned = clean_products()

# Validate emails
invalid_emails = validate_emails()

# Check referential integrity
invalid_order_items = check_referential_integrity()

# Generate report
generate_report()

print("\n" + "=" * 60)
print("DATA CLEANING COMPLETED")
print("=" * 60)

E-COMMERCE DATA CLEANING

Cleaning orders...
Original orders : 1000
Cleaned orders  : 1000

Cleaning products...
Products cleaned: 100

Validating emails...
Invalid emails: 10

Checking referential integrity...
All order_items reference valid orders.

Report created: reports/data_quality_report.txt

DATA CLEANING COMPLETED


## 7. Verify Cleaned Files

In [10]:
print("Cleaned files:")

for file_path in [
    CLEANED_ORDERS_FILE,
    CLEANED_ORDER_ITEMS_FILE,
    CLEANED_PRODUCTS_FILE,
    CLEANED_CUSTOMERS_FILE,
    REPORT_FILE
]:
    print("✓", file_path)

print("\nCleaned Orders:")
display(pd.read_csv(CLEANED_ORDERS_FILE).head())

print("\nCleaned Products:")
display(pd.read_csv(CLEANED_PRODUCTS_FILE).head())

print("\nCleaned Customers:")
display(pd.read_csv(CLEANED_CUSTOMERS_FILE).head())

print("\nCleaned Order Items:")
display(pd.read_csv(CLEANED_ORDER_ITEMS_FILE).head())

Cleaned files:
✓ data/cleaned_orders.csv
✓ data/cleaned_order_items.csv
✓ data/cleaned_products.csv
✓ data/cleaned_customers.csv
✓ reports/data_quality_report.txt

Cleaned Orders:


,order_id,customer_id,order_date,status,region_code
0,ORD00001,CUST0212,2025-12-20 06:26:32,RETURNED,WEST
1,ORD00002,CUST0032,2025-12-28 04:33:13,RETURNED,EAST
2,ORD00003,CUST0193,2025-06-10 05:29:58,RETURNED,EAST
3,ORD00004,CUST0445,2026-01-05 21:41:51,DELIVERED,WEST
4,ORD00005,CUST0286,2025-06-02 07:19:49,DELIVERED,SOUTH



Cleaned Products:


,product_id,product_name,category,subcategory,cost_price
0,PROD0001,Mobile Product 1,Electronics,Mobile,6219.61
1,PROD0002,Mobile Product 2,Electronics,Mobile,42656.03
2,PROD0003,Mobile Product 3,Electronics,Mobile,22656.93
3,PROD0004,Mobile Product 4,Electronics,Mobile,44944.08
4,PROD0005,Mobile Product 5,Electronics,Mobile,22311.05



Cleaned Customers:


,customer_id,customer_name,email,registration_date,customer_type
0,CUST0001,Amit Sharma,amit.sharma1@example.com,2026-01-29,PREMIUM
1,CUST0002,Sneha Kulkarni,sneha.kulkarni2@example.com,2024-05-22,VIP
2,CUST0003,Amit Patil,amit.patil3@example.com,2025-08-27,PREMIUM
3,CUST0004,Rahul Sharma,rahul.sharma4@example.com,2024-04-05,REGULAR
4,CUST0005,Sneha Sharma,sneha.sharma5@example.com,2025-07-28,REGULAR



Cleaned Order Items:


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM000001,ORD00666,PROD0049,3,22311.40,0.58
1,ITEM000002,ORD00120,PROD0011,1,33472.73,25.49
2,ITEM000003,ORD00024,PROD0076,5,24026.97,34.69
3,ITEM000004,ORD00097,PROD0076,2,49997.71,35.14
4,ITEM000005,ORD00306,PROD0074,2,3268.57,45.44
